In [ ]:
#SCRAPPING 
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import time


BASE = "https://www.kurimanzutto.com"
LISTA_URL = "https://www.kurimanzutto.com/artists" 

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; +https://example.org/bot)"
}

def get_artists():
    resp = requests.get(LISTA_URL, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    artists = []
    for a in soup.select("a[href^='/artists']"):
        href = a.get("href")
        name = a.get_text(strip=True)
        if name and href:
            full = urljoin(BASE, href)
            artists.append({"name": name, "url": full})
    df = pd.DataFrame(artists).drop_duplicates(subset=["url"])
    return df

if __name__ == "__main__":
    df = get_artists()
    df.to_csv("kurimanzutto_artists.csv", index=False, encoding="utf-8-sig")
    print(f"Guardados {len(df)} artistas en kurimanzutto_artists.csv")

In [ ]:
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from dotenv import load_dotenv
import urllib.parse
import os

load_dotenv()
EMAIL = os.getenv("ARTPRICE_EMAIL")
PASSWORD = os.getenv("ARTPRICE_PASSWORD")

def rand_sleep(a=0.8, b=1.6):
    time.sleep(random.uniform(a, b))

def safe_click(driver, element):
    driver.execute_script("arguments[0].scrollIntoView(true);", element)
    rand_sleep()
    driver.execute_script("arguments[0].click();", element)

def init_driver():
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    return webdriver.Chrome(options=options)

def login_artprice(driver):
    wait = WebDriverWait(driver, 20)
    driver.get("https://www.artprice.com")
    rand_sleep()
    try:
        login_btn = wait.until(EC.element_to_be_clickable((
            By.XPATH, "//a[contains(@class,'e2e-login-btn')]"
        )))
        safe_click(driver, login_btn)
    except:
        raise Exception("No pude abrir el modal de login (botón e2e-login-btn no disponible).")

    rand_sleep(1.2)

    try:
        email_in = wait.until(EC.presence_of_element_located((
            By.ID, "login"
        )))
        pass_in = wait.until(EC.presence_of_element_located((
            By.ID, "pass"
        )))
    except:
        raise Exception("No encontré input de email o password en el modal.")

    email_in.clear()
    email_in.send_keys(EMAIL)

    pass_in.clear()
    pass_in.send_keys(PASSWORD)
    rand_sleep(0.8)

    try:
        submit_btn = wait.until(EC.element_to_be_clickable((
            By.XPATH, "//button[contains(.,'Log in') or contains(.,'Sign in')]"
        )))
        safe_click(driver, submit_btn)
    except:
        raise Exception("No encontré el botón de iniciar sesión dentro del modal.")

    rand_sleep(2)
    print("Login exitoso.")

def google_find_artist_url(driver, artist_name):
    wait = WebDriverWait(driver, 12)

    driver.get("https://www.google.com/?hl=es")
    rand_sleep(1.3)

    box = wait.until(EC.presence_of_element_located((By.NAME, "q")))
    box.clear()
    query = f"{artist_name} Artprice"
    box.send_keys(query)
    box.send_keys(Keys.ENTER)

    rand_sleep(1.5)

    try:
        link = wait.until(EC.element_to_be_clickable((
            By.XPATH,
            "//a[contains(@href,'artprice.com/artist/')][1]"
        )))
        artist_url = link.get_attribute("href")
        return artist_url
    except:
        return None

def open_past_auctions(driver):
    wait = WebDriverWait(driver, 15)
    rand_sleep()

    try:
        link = wait.until(EC.element_to_be_clickable((
            By.ID, "sln_ps"
        )))
        safe_click(driver, link)
        rand_sleep()
        return True
    except:
        return False
    
def scrape_artwork_list(driver):
    wait = WebDriverWait(driver, 10)

    artworks = []

    cards = driver.find_elements(By.XPATH, "//a[contains(@href,'/artist/') and contains(@href,'/sculpture') or contains(@href,'/drawing') or contains(@href,'/painting')]")

    for card in cards:
        try:
            title = card.text.split("\n")[0].strip()
        except:
            title = ""

        artworks.append({
            "url": card.get_attribute("href"),
            "title": title
        })

    return artworks

def scrape_artwork_page(driver, url):
    wait = WebDriverWait(driver, 12)

    driver.get(url)
    rand_sleep()

    data = {}

    try:
        title = wait.until(EC.presence_of_element_located((
            By.XPATH, "//h1"
        ))).text
        data["title"] = title
    except:
        data["title"] = ""

    try:
        text_blocks = driver.find_elements(By.XPATH, "//div[contains(@class,'lot')]//p | //div[contains(@class,'lot')]//div")
        text = "\n".join([x.text for x in text_blocks if x.text.strip() != ""])
        data["raw_info"] = text
    except:
        data["raw_info"] = ""

    return data

def scrape_artist(driver, artist_name):
    print(f"\n🔎 Scrapeando: {artist_name}")

    url = google_find_artist_url(driver, artist_name)

    if not url:
        print(f"Error scrapeando {artist_name}: No se encontró el artista '{artist_name}' en Google.")
        return []

    driver.get(url)
    rand_sleep()

    if not open_past_auctions(driver):
        print(f"Error scrapeando {artist_name}: No encontré botón 'past auctions'.")
        return []

    art_list = scrape_artwork_list(driver)
    final_data = []

    for art in art_list:
        d = scrape_artwork_page(driver, art["url"])
        d["artist"] = artist_name
        final_data.append(d)

    return final_data

def main():
    driver = init_driver()

    try:
        login_artprice(driver)

        artists_df = pd.read_csv("kurimanzutto_artists.csv")
        artist_list = artists_df["name"].dropna().astype(str).tolist()
        artist_list = [a for a in artist_list if a.lower() != "artists"]

        all_data = []

        for artist in artist_list:
            try:
                data = scrape_artist(driver, artist)
                all_data.extend(data)
            except Exception as e:
                print(f"Error scrapeando artista {artist}: {e}")

        pd.DataFrame(all_data).to_csv("artprice_results.csv", index=False, encoding="utf-8-sig")

        print("\nScraping COMPLETADO, CSV generado: artprice_results.csv")

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import re

df = pd.read_csv('artprice_results.csv', dtype=str, encoding='utf-8')
df = df.fillna("")

def extract(pattern, text, flags=re.IGNORECASE):
    m = re.search(pattern, text, flags)
    return m.group(1).strip() if m else ""

def parse_info(raw):
    tecnica = extract(r"(?:Sculpture-Volume|Installation|Sculpture|Painting|Drawing|Mixed media)[^\n]*\n(.+)", raw)
    dimensiones = extract(r"(\d{1,3}\s*x\s*\d{1,3}(?:\s*x\s*\d{1,3})?\s*cm)", raw)

    estado = "Sold" if re.search(r"\bSold\b", raw, re.IGNORECASE) else "Unsold"

    precio_final = extract(r"Hammer price[:\s]+([^\n]+)", raw)
    if precio_final == "":
        precio_final = extract(r"Sold\s*\n\s*([€$£][^\n]+)", raw)

    precio_estimado = extract(r"Estimate[:\s]+([^\n]+)", raw)

    fecha = extract(r"\b(\d{1,2}\s+[a-zA-Z]{3}\s+\d{4})\b", raw)
    casa = extract(r"\n(Phillips|Sotheby'?s|Christie'?s|Bonhams)\b", raw)
    pais = extract(r",\s*([A-Za-z ]+)\s*$", raw)

    return pd.Series({
        "Fecha": fecha,
        "Técnica": tecnica,
        "Casa": casa,
        "País": pais,
        "Precio estimado": precio_estimado,
        "Precio final": precio_final,
        "Estado de venta": estado,
        "Dimensiones": dimensiones
    })

parsed = df.apply(lambda row: parse_info(row["raw_info"]), axis=1)

parsed["Artista de la obra"] = df["artist"]
parsed["Título de la obra"] = df["title"]

parsed = parsed[
    [
        "Artista de la obra", "Fecha", "Título de la obra", "Técnica",
        "Casa", "País", "Precio estimado", "Precio final",
        "Estado de venta", "Dimensiones"
    ]
].drop_duplicates()

def clean_price(text):
    if not isinstance(text, str) or text.strip() == "":
        return None
    m = re.search(r"€\s*([\d,]+)", text)
    if not m:
        return None
    return float(m.group(1).replace(",", ""))

def clean_price_range(text):
    if not isinstance(text, str) or text.strip() == "":
        return None, None, None
    m = re.findall(r"€\s*([\d,]+)", text)
    if len(m) >= 2:
        low = float(m[0].replace(",", ""))
        high = float(m[1].replace(",", ""))
        avg = (low + high) / 2
        return low, high, avg
    return None, None, None

parsed["estimated_low"], parsed["estimated_high"], parsed["estimated_avg"] = zip(
    *parsed["Precio estimado"].apply(clean_price_range)
)

parsed["final_price"] = parsed["Precio final"].apply(clean_price)

parsed["vendido_flag"] = parsed["final_price"].notna().astype(int)

parsed["final_price_filled"] = parsed["final_price"]

parsed.loc[
    parsed["final_price_filled"].isna() & parsed["estimated_avg"].notna(),
    "final_price_filled"
] = parsed["estimated_avg"]

parsed

In [14]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df = pd.read_csv('artprice_results.csv', dtype=str, encoding='utf-8')
df = df.fillna("")

def extract(pattern, text, flags=re.IGNORECASE):
    m = re.search(pattern, text, flags)
    return m.group(1).strip() if m else ""

def parse_info(raw):
    tecnica = extract(r"(?:Sculpture-Volume|Installation|Sculpture|Painting|Drawing|Mixed media)[^\n]*\n(.+)", raw)
    dimensiones = extract(r"(\d{1,3}\s*x\s*\d{1,3}(?:\s*x\s*\d{1,3})?\s*cm)", raw)

    estado = "Sold" if re.search(r"\bSold\b", raw, re.IGNORECASE) else "Unsold"

    precio_final = extract(r"Hammer price[:\s]+([^\n]+)", raw)
    if precio_final == "":
        precio_final = extract(r"Sold\s*\n\s*([€$£][^\n]+)", raw)

    precio_estimado = extract(r"Estimate[:\s]+([^\n]+)", raw)

    fecha = extract(r"\b(\d{1,2}\s+[a-zA-Z]{3}\s+\d{4})\b", raw)
    casa = extract(r"\n(Phillips|Sotheby'?s|Christie'?s|Bonhams)\b", raw)
    pais = extract(r",\s*([A-Za-z ]+)\s*$", raw)

    return pd.Series({
        "Fecha": fecha,
        "Técnica": tecnica,
        "Casa": casa,
        "País": pais,
        "Precio estimado": precio_estimado,
        "Precio final": precio_final,
        "Estado de venta": estado,
        "Dimensiones": dimensiones
    })


parsed = df.apply(lambda row: parse_info(row["raw_info"]), axis=1)

parsed["Artista de la obra"] = df["artist"]
parsed["Título de la obra"] = df["title"]

parsed = parsed[
    [
        "Artista de la obra", "Fecha", "Título de la obra", "Técnica",
        "Casa", "País", "Precio estimado", "Precio final",
        "Estado de venta", "Dimensiones"
    ]
].drop_duplicates()

def clean_price(text):
    if not isinstance(text, str) or text.strip() == "":
        return None
    m = re.search(r"€\s*([\d,]+)", text)
    if not m:
        return None
    return float(m.group(1).replace(",", ""))

def clean_price_range(text):
    if not isinstance(text, str) or text.strip() == "":
        return None, None, None
    m = re.findall(r"€\s*([\d,]+)", text)
    if len(m) >= 2:
        low = float(m[0].replace(",", ""))
        high = float(m[1].replace(",", ""))
        avg = (low + high) / 2
        return low, high, avg
    return None, None, None

parsed["estimated_low"], parsed["estimated_high"], parsed["estimated_avg"] = zip(
    *parsed["Precio estimado"].apply(clean_price_range)
)

parsed["final_price"] = parsed["Precio final"].apply(clean_price)

parsed["vendido_flag"] = parsed["final_price"].notna().astype(int)

parsed["final_price_filled"] = parsed["final_price"]
parsed.loc[
    parsed["final_price_filled"].isna() & parsed["estimated_avg"].notna(),
    "final_price_filled"
] = parsed["estimated_avg"]


def extract_sale_date(raw):
    if not isinstance(raw, str) or raw.strip() == "":
        return ""

    date_regex = r"(\d{1,2}\s+\w+\s+\d{4}|\d{1,2}/\d{1,2}/\d{4}|\d{4}-\d{2}-\d{2})"
    sale_context = ["sold", "hammer", "hammer price", "sale", "auction", "lot"]

    for key in sale_context:
        pattern = rf"{key}.*?{date_regex}"
        m = re.search(pattern, raw, re.IGNORECASE | re.DOTALL)
        if m:
            return m.group(1).strip()

    for key in sale_context:
        pattern = rf"{date_regex}.*?{key}"
        m = re.search(pattern, raw, re.IGNORECASE | re.DOTALL)
        if m:
            return m.group(1).strip()

    return ""

from dateutil import parser

def normalize_date(text):
    if not isinstance(text, str) or text.strip() == "":
        return None

    try:
        dt = parser.parse(text, dayfirst=True)
        return dt.strftime("%d %b %Y")
    except:
        return None

parsed["Fecha venta"] = df["raw_info"].apply(extract_sale_date)
parsed["Fecha venta"] = parsed["Fecha venta"].apply(normalize_date)

parsed["Fecha venta"] = pd.to_datetime(
    parsed["Fecha venta"],
    format="%d %b %Y",
    errors="coerce"
)

In [18]:
parsed

,Artista de la obra,Fecha,Título de la obra,Técnica,Casa,País,Precio estimado,Precio final,Estado de venta,Dimensiones,estimated_low,estimated_high,estimated_avg,final_price,vendido_flag,final_price_filled,Fecha venta,estimated_price,sold_flag
0,abraham cruzvillegas,10 apr 2025,Definitely Unfinished and Coherent with the La...,"Light installation (plastic roof boards, lumin...",Phillips,,"€ 6,947 - € 9,262 (£ 6,000 - £ 8,000 )",,Sold,184 x 190 x 54 cm,6947.0,9262.0,8104.5,NaN,0,8104.5,2025-04-10,8104.5,1
2,abraham cruzvillegas,07 mar 2025,A Weak Blind Date (2012),"Mixed media sculpture (glass, paint cans, tape...",Phillips,,"€ 7,144 - € 9,526 (£ 6,000 - £ 8,000 )",,Sold,5 x 42 x 67 cm,7144.0,9526.0,8335.0,NaN,0,8335.0,2025-03-07,8335.0,1
4,abraham cruzvillegas,31 oct 2023,Autoconclusión (for Parkett 97) (2015),Multiple (wooden case with 34 coloured bamboo ...,Sotheby's,,"€ 500 - € 1,000",,Sold,,500.0,1000.0,750.0,NaN,0,750.0,2023-10-31,750.0,1
5,abraham cruzvillegas,31 oct 2023,Autoconclusión (for Parkett 97) (2015),Multiple (wooden case with 34 coloured bamboo ...,Phillips,,"€ 500 - € 1,000",,Sold,,500.0,1000.0,750.0,NaN,0,750.0,2023-10-31,750.0,1
6,abraham cruzvillegas,27 sep 2023,Topografos (2003),"Mixed media sculpture (shot glasses, bongos, d...",Phillips,,"€ 11,423 - € 17,134 ($ 12,000 - $ 18,000 )","€ 7,615 ($ 8,000 )",Unsold,,11423.0,17134.0,14278.5,7615.0,1,7615.0,2023-09-27,14278.5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
264,jimmie durham,29 mar 2023,"""Song of the North Bird"" (1965)",Oil/canvas,Christie's,,"€ 9,234 - € 18,468 ($ 10,000 - $ 20,000 )","€ 4,617 ($ 5,000 )",Unsold,80 x 100 cm,9234.0,18468.0,13851.0,4617.0,1,4617.0,2023-03-29,13851.0,1
266,jimmie durham,18 jan 2023,"""1965""",Oil/canvas,Sotheby's,,"€ 18,524 - € 27,786 ($ 20,000 - $ 30,000 )",,Sold,85 x 105 cm,18524.0,27786.0,23155.0,NaN,0,23155.0,2023-01-18,23155.0,1
268,jimmie durham,09 dec 2022,Sans titre (1992),Pencil/paper,,,€ 600 - € 800,€ 650,Unsold,,600.0,800.0,700.0,650.0,1,650.0,2022-12-09,700.0,1
270,jimmie durham,09 dec 2022,Sans titre,Pencil/paper,,,"€ 4,000 - € 6,000","€ 5,000",Unsold,42 x 30 cm,4000.0,6000.0,5000.0,5000.0,1,5000.0,2022-12-09,5000.0,1


In [19]:
import plotly.express as px
import pandas as pd

ventas_reales = parsed[
    (parsed["vendido_flag"] == 0) & 
    (~parsed["final_price_filled"].isna()) &
    (~parsed["Fecha venta"].isna())
].copy()

ventas_reales["Fecha venta"] = pd.to_datetime(
    ventas_reales["Fecha venta"],
    errors="coerce"
)

artistas = ventas_reales["Artista de la obra"].unique()

for artista in artistas:
    df_artista = ventas_reales[ventas_reales["Artista de la obra"] == artista]

    fig = px.scatter(
        df_artista,
        x="Fecha venta",
        y="final_price_filled",
        hover_name="Título de la obra",
        title=f"Ventas de {artista.title()}",
        labels={
            "Fecha venta": "Fecha de venta",
            "final_price_filled": "Cantidad de la venta (€)"
        },
        template="plotly_white"
    )

    fig.update_traces(
        marker=dict(
            size=12,
            line=dict(width=2, color="black")
        )
    )

    fig.update_xaxes(
        tickformat="%d %b %Y",
        showgrid=True,
        nticks=8
    )

    fig.update_layout(
        title_font_size=22,
        xaxis_title_font_size=16,
        yaxis_title_font_size=16,
        plot_bgcolor="rgba(255,255,255,1)"
    )

    fig.show()

In [20]:
import pandas as pd
import numpy as np
import re

df = parsed

def parse_price(val):
    if pd.isna(val) or str(val).strip() == "":
        return np.nan
    
    text = str(val)
    euros = re.findall(r"€\s*([\d,]+)", text)

    if len(euros) == 1:
        return float(euros[0].replace(",", ""))
    elif len(euros) >= 2:
        nums = [float(x.replace(",", "")) for x in euros[:2]]
        return sum(nums) / 2
    return np.nan

df["estimated_price"] = df["Precio estimado"].apply(parse_price)
df["final_price"]     = df["Precio final"].apply(parse_price)

df["sold_flag"] = df["Estado de venta"].str.contains("sold", case=False, na=False).astype(int)

df_complete = df.dropna(subset=["estimated_price", "final_price"])

artist_stats = df_complete.groupby("Artista de la obra").agg({
    "final_price": ["mean", "std", "count"],
    "sold_flag": "mean"
})

artist_stats.columns = [
    "artist_avg_price",
    "artist_price_std",
    "artist_volume",
    "artist_sell_rate"
]

artist_stats = artist_stats.reset_index()

artist_stats["artist_volatility"] = (
    artist_stats["artist_price_std"] / artist_stats["artist_avg_price"]
).replace([np.inf, -np.inf], np.nan)

def minmax(s):
    if s.nunique() <= 1:
        return s * 0
    return (s - s.min()) / (s.max() - s.min())

artist_stats["avg_price_nm"]  = minmax(artist_stats["artist_avg_price"])
artist_stats["sellrate_nm"]   = minmax(artist_stats["artist_sell_rate"])
artist_stats["volume_nm"]     = minmax(np.log1p(artist_stats["artist_volume"]))
artist_stats["volatility_nm"] = 1 - minmax(artist_stats["artist_volatility"])

artist_stats["Artist_Value_Score"] = (
    0.40 * artist_stats["avg_price_nm"] +
    0.25 * artist_stats["sellrate_nm"] +
    0.20 * artist_stats["volume_nm"] +
    0.15 * artist_stats["volatility_nm"]
)

df = df.merge(
    artist_stats[["Artista de la obra", "Artist_Value_Score"]],
    on="Artista de la obra",
    how="left"
)

df_complete = df.dropna(subset=[
    "estimated_price", "final_price", "Artist_Value_Score"
])

gallery_stats = df_complete.groupby("Casa").agg({
    "Artist_Value_Score": "mean",
    "sold_flag": "mean",
    "final_price": "mean",
    "estimated_price": "mean",
    "Título de la obra": "count"
})

gallery_stats.columns = [
    "gallery_artist_score_avg",
    "gallery_sell_through_rate",
    "gallery_avg_final_price",
    "gallery_avg_estimated_price",
    "gallery_volume"
]

gallery_stats["gallery_price_efficiency"] = (
    gallery_stats["gallery_avg_final_price"] /
    gallery_stats["gallery_avg_estimated_price"]
).replace([np.inf, -np.inf], np.nan)

high_threshold = df_complete["final_price"].quantile(0.75)
gallery_stats["gallery_high_value_sales_ratio"] = (
    df_complete.groupby("Casa")["final_price"].apply(lambda x: (x > high_threshold).mean())
)

gallery_stats["artist_nm"] = minmax(gallery_stats["gallery_artist_score_avg"])
gallery_stats["str_nm"]    = minmax(gallery_stats["gallery_sell_through_rate"])
gallery_stats["eff_nm"]    = minmax(gallery_stats["gallery_price_efficiency"])
gallery_stats["high_nm"]   = minmax(gallery_stats["gallery_high_value_sales_ratio"])
gallery_stats["volume_nm"] = minmax(np.log1p(gallery_stats["gallery_volume"]))

gallery_stats["Gallery_Value_Score_raw"] = (
    0.35 * gallery_stats["artist_nm"] +
    0.20 * gallery_stats["str_nm"] +
    0.20 * gallery_stats["eff_nm"] +
    0.15 * gallery_stats["high_nm"] +
    0.10 * gallery_stats["volume_nm"]
)

gallery_stats["Gallery_Value_Score"] = (
    minmax(gallery_stats["Gallery_Value_Score_raw"]) * 100
).round(2)

def label(score):
    if score >= 80:
        return "Elite — Máxima confianza"
    if score >= 60:
        return "Alta — Muy sólida"
    if score >= 40:
        return "Media — Estable"
    return "Baja — Precaución"

gallery_stats["Category"] = gallery_stats["Gallery_Value_Score"].apply(label)

gallery_rank = gallery_stats.sort_values("Gallery_Value_Score", ascending=False)
gallery_rank

,gallery_artist_score_avg,gallery_sell_through_rate,gallery_avg_final_price,gallery_avg_estimated_price,gallery_volume,gallery_price_efficiency,gallery_high_value_sales_ratio,artist_nm,str_nm,eff_nm,high_nm,volume_nm,Gallery_Value_Score_raw,Gallery_Value_Score,Category
Casa,,,,,,,,,,,,,,,
Sotheby's,0.355302,1.0,35594.642857,28491.821429,14,1.249293,0.357143,0.899105,0.0,0.589132,1.000000,0.735141,0.656027,100.00,Elite — Máxima confianza
Phillips,0.356763,1.0,14049.933333,13654.766667,30,1.028940,0.300000,0.927492,0.0,0.300209,0.840000,1.000000,0.610664,89.80,Elite — Máxima confianza
,0.337134,1.0,3339.875000,2137.312500,8,1.562652,0.000000,0.545994,0.0,1.000000,0.000000,0.548765,0.445974,52.79,Media — Estable
Bonhams,0.360493,1.0,7473.000000,9341.500000,1,0.799979,0.000000,1.000000,0.0,0.000000,0.000000,0.000000,0.350000,31.21,Baja — Precaución
Christie's,0.309042,1.0,9703.333333,11001.518519,27,0.881999,0.222222,0.000000,0.0,0.107544,0.622222,0.962864,0.211129,0.00,Baja — Precaución
